# Notebook 04 — Elasticsearch Appendix (Optional Homework)

**Workshop:** Agentic AI for Scientists — Week 2
**Status:** Optional — not run in class. Self-paced after.
**Prereq:** Notebook 03 working (we reuse the corpus + chunks + eval set).

**Goal:** Show how the in-memory hybrid pipeline from Notebook 03 looks in production. We use **Elasticsearch** with native **RRF (Reciprocal Rank Fusion)** — the standard hybrid-search recipe in production ES. It's the same RRF that `EnsembleRetriever` used in Notebook 03, now running inside a real search engine.

**You need:**
- A free Elastic Cloud trial (14 days, no credit card): https://cloud.elastic.co
- Your `ELASTIC_CLOUD_ID` and `ELASTIC_API_KEY` in `.env`

**What changes vs Notebook 03:**

| | In-memory (NB03) | Elasticsearch (NB04) |
|---|---|---|
| Dense store | FAISS / Chroma (in RAM / local disk) | ES `dense_vector` field with HNSW index |
| Sparse store | rank_bm25 (in RAM) | ES inverted index with BM25 scoring |
| Hybrid | EnsembleRetriever (RRF) | Native RRF retriever |
| Scale | ~10k chunks before RAM hurts | Millions of chunks, sharded |
| Filters | Manual Python | Native query DSL (date, source, metadata) |

The retrieval *interface* is the same: `retrieve_X(query, k)` returns chunks. Plug it into the same RAG chain from Notebook 03.


## 0. Setup

In [ ]:
%pip install -q "elasticsearch==8.15.1" "sentence-transformers==3.3.0" "pypdf==5.1.0" \
    "pandas==2.2.3" "anthropic>=0.39.0,<0.50.0" python-dotenv requests


In [ ]:
import os, json, re
from pathlib import Path
import requests
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch, helpers

try:
    from google.colab import userdata  # type: ignore
    os.environ["ELASTIC_CLOUD_ID"] = userdata.get("ELASTIC_CLOUD_ID")
    os.environ["ELASTIC_API_KEY"] = userdata.get("ELASTIC_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

CLOUD_ID = os.environ.get("ELASTIC_CLOUD_ID")
API_KEY = os.environ.get("ELASTIC_API_KEY")
assert CLOUD_ID and API_KEY, (
    "Set ELASTIC_CLOUD_ID and ELASTIC_API_KEY first. Free trial: https://cloud.elastic.co"
)

es = Elasticsearch(cloud_id=CLOUD_ID, api_key=API_KEY, request_timeout=60)
print(f"Connected to ES: {es.info()['cluster_name']}")


---

## 1. Recreate the corpus + chunks (reuses NB03 logic)

In [ ]:
CORPUS_DIR = Path("sample_articles")
CORPUS_DIR.mkdir(exist_ok=True)
PAPERS = {
    "react_yao_2022.pdf": "https://arxiv.org/pdf/2210.03629",
    "attention_vaswani_2017.pdf": "https://arxiv.org/pdf/1706.03762",
    "chinchilla_hoffmann_2022.pdf": "https://arxiv.org/pdf/2203.15556",
    "constitutional_ai_bai_2022.pdf": "https://arxiv.org/pdf/2212.08073",
    "rag_lewis_2020.pdf": "https://arxiv.org/pdf/2005.11401",
}
for filename, url in PAPERS.items():
    target = CORPUS_DIR / filename
    if not target.exists():
        target.write_bytes(requests.get(url, timeout=60, headers={"User-Agent": "agentic-workshop/1.0"}).content)
        print(f"  downloaded {filename}")
print("Corpus ready.")


In [ ]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100,
                                          separators=["\n\n", "\n", ". ", " ", ""])

chunks = []
for path in sorted(CORPUS_DIR.glob("*.pdf")):
    text = "\n\n".join(p.extract_text() or "" for p in PdfReader(str(path)).pages)
    for i, piece in enumerate(splitter.split_text(text)):
        chunks.append({"text": piece, "source": path.name, "chunk_id": f"{path.name}#{i}"})

print(f"{len(chunks)} chunks across {len(PAPERS)} papers")

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
EMB_DIM = embedder.get_sentence_embedding_dimension()
print(f"Embedding chunks (dim={EMB_DIM})...")
embs = embedder.encode([c["text"] for c in chunks], show_progress_bar=True, normalize_embeddings=True)
print(f"  shape={embs.shape}")


---

## 2. Create the ES index with mixed mapping

One field for the dense vector, one for the text (BM25). Both live in the same document.


In [ ]:
INDEX = "agentic-w2-papers"

if es.indices.exists(index=INDEX):
    print(f"  deleting existing index {INDEX}")
    es.indices.delete(index=INDEX)

es.indices.create(
    index=INDEX,
    mappings={
        "properties": {
            "text": {"type": "text"},
            "source": {"type": "keyword"},
            "chunk_id": {"type": "keyword"},
            "embedding": {"type": "dense_vector", "dims": EMB_DIM, "index": True, "similarity": "cosine"},
        }
    },
)
print(f"Created index {INDEX}")


---

## 3. Bulk-index the chunks

In [ ]:
actions = [
    {"_index": INDEX, "_id": c["chunk_id"], "_source": {**c, "embedding": emb.tolist()}}
    for c, emb in zip(chunks, embs)
]
helpers.bulk(es, actions, request_timeout=60)
es.indices.refresh(index=INDEX)
print(f"Indexed {es.count(index=INDEX)['count']} chunks")


---

## 4. Dense, sparse, and hybrid retrieval against ES

Same `retrieve_X(query, k)` shape as Notebook 03 — different backend.


In [ ]:
def retrieve_dense_es(query, k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    resp = es.search(index=INDEX, size=k,
                     knn={"field": "embedding", "query_vector": q_emb, "k": k, "num_candidates": 50},
                     source_includes=["text", "source", "chunk_id"])
    return [{**h["_source"], "score": h["_score"]} for h in resp["hits"]["hits"]]


def retrieve_bm25_es(query, k=5):
    resp = es.search(index=INDEX, size=k, query={"match": {"text": query}},
                     source_includes=["text", "source", "chunk_id"])
    return [{**h["_source"], "score": h["_score"]} for h in resp["hits"]["hits"]]


def retrieve_hybrid_rrf(query, k=5, rank_constant=60):
    """Native RRF via the ES retrievers API (ES 8.13+)."""
    q_emb = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    resp = es.search(index=INDEX, size=k, retriever={
        "rrf": {
            "retrievers": [
                {"standard": {"query": {"match": {"text": query}}}},
                {"knn": {"field": "embedding", "query_vector": q_emb, "k": k, "num_candidates": 50}},
            ],
            "rank_window_size": 50,
            "rank_constant": rank_constant,
        }
    }, source_includes=["text", "source", "chunk_id"])
    return [{**h["_source"], "score": h["_score"]} for h in resp["hits"]["hits"]]


for name, fn in [("dense", retrieve_dense_es), ("bm25", retrieve_bm25_es), ("rrf-hybrid", retrieve_hybrid_rrf)]:
    print(f"\n--- {name} ---")
    for r in fn("What is the ReAct loop?", k=3):
        print(f"  [{r['score']:.3f}] {r['chunk_id']}: {r['text'][:100]}...")


---

## 5. Same comparison as NB03, against ES

In [ ]:
# Same five labelled questions as Notebook 03, inlined (no external dependency).
questions = [
    {"id": "q1_react_loop",
     "question": "What are the three steps in a single iteration of the ReAct loop?",
     "expected_source": "react_yao_2022.pdf",
     "expected_section_keywords": ["Thought", "Action", "Observation", "interleaving"]},
    {"id": "q2_attention_complexity",
     "question": "Why is scaled dot-product attention preferred over additive attention for transformer architectures?",
     "expected_source": "attention_vaswani_2017.pdf",
     "expected_section_keywords": ["scaled dot-product", "additive", "matrix multiplication", "faster"]},
    {"id": "q3_chinchilla_scaling",
     "question": "According to the Chinchilla paper, what is the optimal ratio of training tokens to model parameters?",
     "expected_source": "chinchilla_hoffmann_2022.pdf",
     "expected_section_keywords": ["tokens per parameter", "compute-optimal", "scaling", "training tokens"]},
    {"id": "q4_constitutional_ai_rlaif",
     "question": "What is the role of the AI feedback signal in Constitutional AI training?",
     "expected_source": "constitutional_ai_bai_2022.pdf",
     "expected_section_keywords": ["RLAIF", "AI feedback", "preference model", "constitution", "self-critique"]},
    {"id": "q5_rag_architecture",
     "question": "What are the two main components of the retrieval-augmented generation architecture?",
     "expected_source": "rag_lewis_2020.pdf",
     "expected_section_keywords": ["retriever", "generator", "non-parametric", "parametric", "BART"]},
]


def hit_at_k(retrieved, expected_source):
    return int(any(r["source"] == expected_source for r in retrieved))


def section_hit(retrieved, keywords):
    blob = " ".join(r["text"].lower() for r in retrieved)
    return int(any(kw.lower() in blob for kw in keywords))


rows = []
for q in questions:
    de = retrieve_dense_es(q["question"], k=5)
    bm = retrieve_bm25_es(q["question"], k=5)
    rr = retrieve_hybrid_rrf(q["question"], k=5)
    rows.append({
        "id": q["id"], "expected": q["expected_source"].split("_")[0],
        "es_dense_hit@5": hit_at_k(de, q["expected_source"]),
        "es_bm25_hit@5": hit_at_k(bm, q["expected_source"]),
        "es_rrf_hit@5": hit_at_k(rr, q["expected_source"]),
        "es_dense_section": section_hit(de, q["expected_section_keywords"]),
        "es_bm25_section": section_hit(bm, q["expected_section_keywords"]),
        "es_rrf_section": section_hit(rr, q["expected_section_keywords"]),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\n=== Aggregate ===")
for col in df.columns[2:]:
    print(f"  {col:22s}: {df[col].mean():.2f}")


**Compare these numbers to Notebook 03's table.** On our 5-paper corpus the results are similar — ES doesn't *retrieve better* at this scale. The win shows up when:

- Corpus grows past ~10k chunks (RAM becomes the limit for FAISS)
- You need metadata filtering (date ranges, source authorities)
- You need cross-cluster replication, sharding, or write throughput

**But the principle is identical.** Dense + sparse + RRF — whether the fusion runs in `EnsembleRetriever` or in Elasticsearch is a deployment detail.

---

## 6. Cleanup


In [ ]:
# Uncomment to delete the index when you're done (so you don't use trial storage)
# es.indices.delete(index=INDEX)
# print(f"Deleted {INDEX}")


---

## 7. Where to go from here

- **Weaviate** — same hybrid pattern, open-source, easy local Docker
- **Qdrant** — vector-first, very fast
- **pgvector** — if you're already on Postgres
- **Vespa** — Yahoo-grade scale, complex setup

All implement the same "dense + sparse + fusion" recipe. Once you've built it in-memory (NB03) and in ES (NB04), swapping vendors is a small adapter.
